In [ ]:
from __future__ import annotations

import re
import shutil
from collections import defaultdict
from pathlib import Path
from typing import Any

import arcpy
from arcpy import env

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side


# =============================================================================
# CONFIG
# =============================================================================

Previous_STAMP = "2026-01"
ROUNDSTAMP = "2026-06"

BASE_DIR = Path(r"D:\A02-Projects\Shapefile_Cleaning")

PREVIOUS_WORKSPACE = BASE_DIR / "02_archive" / Previous_STAMP
RAW_WORKSPACE = BASE_DIR / "00_raw" / ROUNDSTAMP
ARCHIVE_DIR = BASE_DIR / "02_archive" / ROUNDSTAMP

DRY_RUN = False
# True  = create report only: do not delete duplicate polygons, do not write NOTICE,
#         do not delete empty shapefiles, and do not copy surviving shapefiles to ARCHIVE_DIR
# False = actually delete exact duplicate polygons, write NOTICE, delete shapefiles left with
#         0 polygons, and copy the surviving (non-empty) shapefiles to ARCHIVE_DIR

ARCHIVE_OVERWRITE = False
# True  = replace existing shapefile group in ARCHIVE_DIR
# False = skip archive copy if target shapefile group already exists

DEL_PREV_REPORT = (
    BASE_DIR
    / "05_reports"
    / f"{ROUNDSTAMP}"
    / f"04_{ROUNDSTAMP}-Del_Previous.xlsx"
)
DEL_PREV_REPORT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# FIELD CONFIG
# =============================================================================

GEOMETRY_COMPARE_FIELDS = [
    "INSIDE_X",
    "INSIDE_Y",
    "AREA",
    "PERIMETER",
]

APID_FIELD = "APID"
NOTICE_FIELD = "NOTICE"
NOTICE_TEXT = "พบรูปแปลงซ้ำกับรอบก่อน แต่ระวางแผนที่ไม่ตรงกัน"

# Calculated geometry fields can have tiny floating differences.
# Use 6 decimals unless you need strict exact comparison.
FLOAT_ROUND_DIGITS = 6


# =============================================================================
# BASIC HELPERS
# =============================================================================

def normalize_text(value: Any) -> str:
    """
    Convert value to clean text.
    """
    if value is None:
        return ""

    text = str(value).replace("\u00a0", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_apid(value: Any) -> str:
    """
    Normalize APID for comparison.
    """
    return normalize_text(value)


def normalize_float_key(value: Any) -> float | str:
    """
    Normalize geometry values for duplicate comparison.

    Example:
        123.123456789 -> 123.123457
    """
    if value is None:
        return ""

    try:
        return round(float(value), FLOAT_ROUND_DIGITS)
    except Exception:
        return normalize_text(value)


def make_geometry_key(values: list[Any]) -> tuple[Any, ...]:
    """
    Build duplicate geometry key from:
        INSIDE_X, INSIDE_Y, AREA, PERIMETER
    """
    return tuple(normalize_float_key(value) for value in values)


def make_exact_key(geometry_key: tuple[Any, ...], apid: Any) -> tuple[Any, ...]:
    """
    Build exact duplicate key from:
        INSIDE_X, INSIDE_Y, AREA, PERIMETER, APID
    """
    return geometry_key + (normalize_apid(apid),)


def find_shapefiles(root_dir: Path) -> list[Path]:
    """
    Find every .shp under a workspace.
    """
    if not root_dir.exists():
        raise FileNotFoundError(f"Workspace not found: {root_dir}")

    return sorted(
        path
        for path in root_dir.rglob("*")
        if path.is_file() and path.suffix.lower() == ".shp"
    )


def get_field_map(feature_class: Path | str) -> dict[str, arcpy.Field]:
    """
    Return field map using uppercase field names.
    """
    return {
        field.name.upper(): field
        for field in arcpy.ListFields(str(feature_class))
    }


def get_actual_field_name(feature_class: Path | str, field_name: str) -> str | None:
    """
    Return actual field name, case-insensitive.
    """
    field_map = get_field_map(feature_class)
    field = field_map.get(field_name.upper())

    if field is None:
        return None

    return field.name


def validate_fields(feature_class: Path, required_fields: list[str]) -> list[str]:
    """
    Return missing field names.
    """
    field_map = get_field_map(feature_class)

    missing = []

    for field_name in required_fields:
        if field_name.upper() not in field_map:
            missing.append(field_name)

    return missing


def add_text_field_if_missing(
    feature_class: Path,
    field_name: str,
    field_length: int = 254,
) -> str:
    """
    Add text field if missing.

    NOTICE is safe for shapefile because the field name has fewer than 10 characters.
    """
    field_map = get_field_map(feature_class)

    if field_name.upper() in field_map:
        return "EXISTS"

    if DRY_RUN:
        return "DRY_RUN_ADD"

    arcpy.management.AddField(
        in_table=str(feature_class),
        field_name=field_name,
        field_type="TEXT",
        field_length=field_length,
    )

    return "ADDED"


def build_shapefile_index(shapefiles: list[Path]) -> dict[str, list[Path]]:
    """
    Build index:

        shapefile stem -> list of shapefile paths

    Example:
        {
            "10000000_47": [
                Path(...10000000_47.shp)
            ]
        }
    """
    index: dict[str, list[Path]] = defaultdict(list)

    for shp_path in shapefiles:
        index[shp_path.stem].append(shp_path)

    return index


# =============================================================================
# LOAD PREVIOUS DATA
# =============================================================================

def load_previous_keys(previous_shp_paths: list[Path]) -> tuple[set[tuple[Any, ...]], dict[tuple[Any, ...], set[str]], str]:
    """
    Load comparison keys from previous shapefile(s).

    Returns:
        previous_exact_keys:
            Set of:
                INSIDE_X, INSIDE_Y, AREA, PERIMETER, APID

        previous_geometry_to_apids:
            Dict:
                geometry key -> set of APID values

        message:
            Processing note
    """
    required_fields = GEOMETRY_COMPARE_FIELDS + [APID_FIELD]

    previous_exact_keys: set[tuple[Any, ...]] = set()
    previous_geometry_to_apids: dict[tuple[Any, ...], set[str]] = defaultdict(set)

    messages = []

    for previous_shp in previous_shp_paths:
        missing = validate_fields(previous_shp, required_fields)

        if missing:
            messages.append(
                f"Previous missing fields in {previous_shp.name}: {', '.join(missing)}"
            )
            continue

        actual_fields = [
            get_actual_field_name(previous_shp, field_name)
            for field_name in required_fields
        ]

        with arcpy.da.SearchCursor(str(previous_shp), actual_fields) as cursor:
            for row in cursor:
                geometry_values = list(row[:4])
                apid_value = row[4]

                if any(value is None for value in geometry_values):
                    continue

                geometry_key = make_geometry_key(geometry_values)
                apid = normalize_apid(apid_value)
                exact_key = make_exact_key(geometry_key, apid)

                previous_exact_keys.add(exact_key)
                previous_geometry_to_apids[geometry_key].add(apid)

    if not messages:
        messages.append("Previous keys loaded")

    return previous_exact_keys, previous_geometry_to_apids, " | ".join(messages)


# =============================================================================
# PROCESS CURRENT SHAPEFILE
# =============================================================================

def process_current_shapefile(
    current_shp: Path,
    previous_shp_paths: list[Path],
) -> dict[str, Any]:
    """
    Compare one current shapefile against previous shapefile(s).

    Rule 1.1:
        If INSIDE_X, INSIDE_Y, AREA, PERIMETER, APID all match:
            delete current polygon.

    Rule 1.2:
        If INSIDE_X, INSIDE_Y, AREA, PERIMETER match
        but APID is different:
            write NOTICE in current polygon.
    """
    result = {
        "shapefile name": current_shp.stem,
        "Count of Delete": 0,
        "Count of Notice": 0,
        "status": "",
        "message": "",
        "current_path": str(current_shp),
        "previous_path_count": len(previous_shp_paths),
        "previous_paths": " | ".join(str(path) for path in previous_shp_paths),
    }

    try:
        if not previous_shp_paths:
            result["status"] = "SKIP"
            result["message"] = "No matching previous shapefile found"
            return result

        required_current_fields = GEOMETRY_COMPARE_FIELDS + [APID_FIELD]
        missing_current = validate_fields(current_shp, required_current_fields)

        if missing_current:
            result["status"] = "ERROR"
            result["message"] = f"Current missing fields: {', '.join(missing_current)}"
            return result

        previous_exact_keys, previous_geometry_to_apids, previous_message = load_previous_keys(
            previous_shp_paths
        )

        if not previous_exact_keys and not previous_geometry_to_apids:
            result["status"] = "SKIP"
            result["message"] = f"No usable previous keys loaded. {previous_message}"
            return result

        add_notice_status = add_text_field_if_missing(
            feature_class=current_shp,
            field_name=NOTICE_FIELD,
            field_length=254,
        )

        actual_current_fields = [
            get_actual_field_name(current_shp, field_name)
            for field_name in GEOMETRY_COMPARE_FIELDS + [APID_FIELD, NOTICE_FIELD]
        ]

        cursor_fields = ["OID@"] + actual_current_fields

        delete_count = 0
        notice_count = 0
        skipped_null_geometry_count = 0

        if DRY_RUN:
            with arcpy.da.SearchCursor(str(current_shp), cursor_fields) as cursor:
                for row in cursor:
                    geometry_values = list(row[1:5])
                    apid_value = row[5]

                    if any(value is None for value in geometry_values):
                        skipped_null_geometry_count += 1
                        continue

                    geometry_key = make_geometry_key(geometry_values)
                    apid = normalize_apid(apid_value)
                    exact_key = make_exact_key(geometry_key, apid)

                    if exact_key in previous_exact_keys:
                        delete_count += 1

                    elif geometry_key in previous_geometry_to_apids:
                        previous_apids = previous_geometry_to_apids[geometry_key]

                        if apid not in previous_apids:
                            notice_count += 1

        else:
            with arcpy.da.UpdateCursor(str(current_shp), cursor_fields) as cursor:
                for row in cursor:
                    geometry_values = list(row[1:5])
                    apid_value = row[5]

                    if any(value is None for value in geometry_values):
                        skipped_null_geometry_count += 1
                        continue

                    geometry_key = make_geometry_key(geometry_values)
                    apid = normalize_apid(apid_value)
                    exact_key = make_exact_key(geometry_key, apid)

                    if exact_key in previous_exact_keys:
                        cursor.deleteRow()
                        delete_count += 1

                    elif geometry_key in previous_geometry_to_apids:
                        previous_apids = previous_geometry_to_apids[geometry_key]

                        if apid not in previous_apids:
                            row[6] = NOTICE_TEXT
                            cursor.updateRow(row)
                            notice_count += 1

        result["Count of Delete"] = delete_count
        result["Count of Notice"] = notice_count

        if DRY_RUN:
            result["status"] = "DRY_RUN"
            result["message"] = (
                f"Checked only. NOTICE field status={add_notice_status}. "
                f"Skipped null geometry={skipped_null_geometry_count}. "
                f"{previous_message}"
            )
        else:
            result["status"] = "SUCCESS"
            result["message"] = (
                f"Deleted exact duplicates and wrote NOTICE. "
                f"NOTICE field status={add_notice_status}. "
                f"Skipped null geometry={skipped_null_geometry_count}. "
                f"{previous_message}"
            )

        return result

    except Exception as exc:
        result["status"] = "ERROR"
        result["message"] = repr(exc)
        return result


# =============================================================================
# POST-PROCESS: DELETE EMPTY SHAPEFILES, ARCHIVE THE SURVIVING SHAPEFILES
# =============================================================================

def get_feature_count(shp_path: Path) -> int:
    """
    Return feature count for a shapefile.
    """
    return int(arcpy.management.GetCount(str(shp_path))[0])


def find_shapefile_group(shp_path: Path) -> list[Path]:
    """
    Find all files belonging to one shapefile group.

    Example:
        10000000_47.shp
        10000000_47.shx
        10000000_47.dbf
        10000000_47.prj
        10000000_47.cpg
        10000000_47.shp.xml

    Lock files are excluded.
    """
    return sorted(
        path
        for path in shp_path.parent.glob(f"{shp_path.stem}.*")
        if path.is_file() and not path.name.lower().endswith(".lock")
    )


def find_archive_group(stem: str) -> list[Path]:
    """
    Find existing archived shapefile group files by stem.
    """
    return sorted(
        path
        for path in ARCHIVE_DIR.glob(f"{stem}.*")
        if path.is_file()
    )


def check_empty_shapefile(shp_path: Path) -> dict[str, Any]:
    """
    Check whether a shapefile has no polygons left.

    This runs after the previous-duplicate delete process.
    """
    result = {
        "shapefile name": shp_path.stem,
        "full path": str(shp_path),
        "feature_count": "",
        "is_empty": "",
        "status": "",
        "message": "",
    }

    try:
        if not arcpy.Exists(str(shp_path)):
            result["status"] = "ERROR"
            result["message"] = "Shapefile does not exist or ArcPy cannot read it"
            return result

        feature_count = get_feature_count(shp_path)

        result["feature_count"] = feature_count
        result["is_empty"] = "YES" if feature_count == 0 else "NO"
        result["status"] = "EMPTY" if feature_count == 0 else "NOT_EMPTY"
        result["message"] = "No polygon left" if feature_count == 0 else "Shapefile still has polygons"
        return result

    except Exception as exc:
        result["status"] = "ERROR"
        result["message"] = repr(exc)
        return result


def delete_empty_shapefile_group(shp_path: Path) -> dict[str, Any]:
    """
    Delete an empty shapefile group (0 polygons remaining) from RAW_WORKSPACE.

    Empty shapefiles hold nothing of value after the previous-duplicate delete
    process (every polygon they had was an exact duplicate), so they are removed
    outright rather than archived.
    """
    source_group_files = find_shapefile_group(shp_path)

    result = {
        "shapefile name": shp_path.stem,
        "source folder": str(shp_path.parent),
        "source file count": len(source_group_files),
        "deleted file count": 0,
        "source files": " | ".join(path.name for path in source_group_files),
        "status": "",
        "message": "",
    }

    try:
        if not source_group_files:
            result["status"] = "ERROR"
            result["message"] = "No shapefile group files found to delete"
            return result

        if DRY_RUN:
            result["status"] = "DRY_RUN_DELETE"
            result["message"] = "Checked only, empty shapefile group not deleted"
            result["deleted file count"] = len(source_group_files)
            return result

        deleted_count = 0

        for source_file in source_group_files:
            source_file.unlink()
            deleted_count += 1

        result["deleted file count"] = deleted_count
        result["status"] = "DELETED"
        result["message"] = "Empty shapefile group deleted from RAW_WORKSPACE"
        return result

    except PermissionError:
        result["status"] = "ERROR"
        result["message"] = "Permission denied. File may be open or locked."
        return result

    except Exception as exc:
        result["status"] = "ERROR"
        result["message"] = repr(exc)
        return result


def archive_surviving_shapefile_group(shp_path: Path, copied_stems: set[str]) -> dict[str, Any]:
    """
    Copy one surviving (non-empty) shapefile group from RAW_WORKSPACE to ARCHIVE_DIR.

    These are the shapefiles that still have polygons after the previous-duplicate
    delete process - i.e. the cleaned, final data for this round. RAW_WORKSPACE is
    left untouched; this is a copy, not a move.

    The archive is flat:
        ARCHIVE_DIR / 10000000_47.shp
        ARCHIVE_DIR / 10000000_47.dbf
        ARCHIVE_DIR / 10000000_47.shx
        ...

    If duplicate stems are found from different source subfolders, only the first is handled.
    """
    source_group_files = find_shapefile_group(shp_path)
    archive_group_files = find_archive_group(shp_path.stem)

    result = {
        "shapefile name": shp_path.stem,
        "source folder": str(shp_path.parent),
        "archive folder": str(ARCHIVE_DIR),
        "source file count": len(source_group_files),
        "archive existing file count": len(archive_group_files),
        "copied file count": 0,
        "source files": " | ".join(path.name for path in source_group_files),
        "archive existing files": " | ".join(path.name for path in archive_group_files),
        "status": "",
        "message": "",
    }

    try:
        if shp_path.stem in copied_stems:
            result["status"] = "SKIP"
            result["message"] = "Duplicate shapefile stem already handled"
            return result

        copied_stems.add(shp_path.stem)

        if not source_group_files:
            result["status"] = "ERROR"
            result["message"] = "No shapefile group files found to copy"
            return result

        if archive_group_files and not ARCHIVE_OVERWRITE:
            result["status"] = "SKIP"
            result["message"] = "Archive target already exists. Set ARCHIVE_OVERWRITE = True to replace it."
            return result

        if DRY_RUN:
            result["status"] = "DRY_RUN_COPY"
            result["message"] = "Checked only, shapefile group not copied"
            result["copied file count"] = len(source_group_files)
            return result

        if archive_group_files and ARCHIVE_OVERWRITE:
            for archive_file in archive_group_files:
                archive_file.unlink()

        copied_count = 0

        for source_file in source_group_files:
            target_file = ARCHIVE_DIR / source_file.name
            shutil.copy2(source_file, target_file)
            copied_count += 1

        result["copied file count"] = copied_count
        result["status"] = "COPIED"
        result["message"] = "Surviving shapefile group copied to archive"
        return result

    except PermissionError:
        result["status"] = "ERROR"
        result["message"] = "Permission denied. File may be open or locked."
        return result

    except Exception as exc:
        result["status"] = "ERROR"
        result["message"] = repr(exc)
        return result


# =============================================================================
# EXCEL REPORT
# =============================================================================

def style_worksheet(ws) -> None:
    """
    Apply basic Excel formatting.
    """
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)

    thin_border = Border(
        left=Side(style="thin", color="D9E2F3"),
        right=Side(style="thin", color="D9E2F3"),
        top=Side(style="thin", color="D9E2F3"),
        bottom=Side(style="thin", color="D9E2F3"),
    )

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border

    for row in ws.iter_rows():
        for cell in row:
            cell.border = thin_border
            cell.alignment = Alignment(vertical="top", wrap_text=True)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions


def autosize_columns(ws, max_width: int = 100) -> None:
    """
    Auto-size Excel columns.
    """
    for column_cells in ws.columns:
        max_length = 0
        column_letter = column_cells[0].column_letter

        for cell in column_cells:
            value = cell.value

            if value is None:
                continue

            max_length = max(max_length, len(str(value)))

        ws.column_dimensions[column_letter].width = min(max_length + 2, max_width)


def write_sheet(ws, headers: list[str], rows: list[dict[str, Any]]) -> None:
    """
    Write rows to one worksheet.
    """
    ws.append(headers)

    for row in rows:
        ws.append([row.get(header, "") for header in headers])

    style_worksheet(ws)
    autosize_columns(ws)


def write_report_excel(
    report_rows: list[dict[str, Any]],
    empty_rows: list[dict[str, Any]],
    delete_rows: list[dict[str, Any]],
    archive_rows: list[dict[str, Any]],
) -> None:
    """
    Write Excel report to DEL_PREV_REPORT.
    """
    wb = Workbook()

    ws = wb.active
    ws.title = "del_pre"

    write_sheet(
        ws=ws,
        headers=[
            "shapefile name",
            "Count of Delete",
            "Count of Notice",
            "status",
            "message",
            "current_path",
            "previous_path_count",
            "previous_paths",
        ],
        rows=report_rows,
    )

    ws_empty = wb.create_sheet("empty_check")

    write_sheet(
        ws=ws_empty,
        headers=[
            "shapefile name",
            "full path",
            "feature_count",
            "is_empty",
            "status",
            "message",
        ],
        rows=empty_rows,
    )

    ws_delete = wb.create_sheet("delete_empty")

    write_sheet(
        ws=ws_delete,
        headers=[
            "shapefile name",
            "source folder",
            "source file count",
            "deleted file count",
            "source files",
            "status",
            "message",
        ],
        rows=delete_rows,
    )

    ws_archive = wb.create_sheet("archive_survivors")

    write_sheet(
        ws=ws_archive,
        headers=[
            "shapefile name",
            "source folder",
            "archive folder",
            "source file count",
            "archive existing file count",
            "copied file count",
            "source files",
            "archive existing files",
            "status",
            "message",
        ],
        rows=archive_rows,
    )

    ws_summary = wb.create_sheet("Summary")
    ws_summary.append(["Item", "Value"])

    summary_rows = [
        ["Previous_STAMP", Previous_STAMP],
        ["ROUNDSTAMP", ROUNDSTAMP],
        ["PREVIOUS_WORKSPACE", str(PREVIOUS_WORKSPACE)],
        ["RAW_WORKSPACE", str(RAW_WORKSPACE)],
        ["ARCHIVE_DIR", str(ARCHIVE_DIR)],
        ["DRY_RUN", DRY_RUN],
        ["ARCHIVE_OVERWRITE", ARCHIVE_OVERWRITE],
        ["DEL_PREV_REPORT", str(DEL_PREV_REPORT)],
        ["Total shapefiles processed", len(report_rows)],
        ["Total Count of Delete", sum(int(row.get("Count of Delete") or 0) for row in report_rows)],
        ["Total Count of Notice", sum(int(row.get("Count of Notice") or 0) for row in report_rows)],
        ["SUCCESS", sum(1 for row in report_rows if row.get("status") == "SUCCESS")],
        ["DRY_RUN rows", sum(1 for row in report_rows if row.get("status") == "DRY_RUN")],
        ["SKIP", sum(1 for row in report_rows if row.get("status") == "SKIP")],
        ["ERROR", sum(1 for row in report_rows if row.get("status") == "ERROR")],
        ["Empty check rows", len(empty_rows)],
        ["Empty shapefiles found", sum(1 for row in empty_rows if row.get("is_empty") == "YES")],
        ["Delete rows", len(delete_rows)],
        ["Delete DELETED", sum(1 for row in delete_rows if row.get("status") == "DELETED")],
        ["Delete DRY_RUN_DELETE", sum(1 for row in delete_rows if row.get("status") == "DRY_RUN_DELETE")],
        ["Delete ERROR", sum(1 for row in delete_rows if row.get("status") == "ERROR")],
        ["Archive copy rows (survivors)", len(archive_rows)],
        ["Archive COPIED", sum(1 for row in archive_rows if row.get("status") == "COPIED")],
        ["Archive DRY_RUN_COPY", sum(1 for row in archive_rows if row.get("status") == "DRY_RUN_COPY")],
        ["Archive SKIP", sum(1 for row in archive_rows if row.get("status") == "SKIP")],
        ["Archive ERROR", sum(1 for row in archive_rows if row.get("status") == "ERROR")],
        ["Total archive files copied / would copy", sum(int(row.get("copied file count") or 0) for row in archive_rows)],
    ]

    for row in summary_rows:
        ws_summary.append(row)

    style_worksheet(ws_summary)
    autosize_columns(ws_summary)

    wb.save(DEL_PREV_REPORT)


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    env.overwriteOutput = True

    print("============================================================")
    print("Delete previous duplicated polygons / write NOTICE / delete empty shapefiles / archive survivors")
    print("============================================================")
    print(f"Previous_STAMP     : {Previous_STAMP}")
    print(f"ROUNDSTAMP         : {ROUNDSTAMP}")
    print(f"PREVIOUS_WORKSPACE : {PREVIOUS_WORKSPACE}")
    print(f"RAW_WORKSPACE      : {RAW_WORKSPACE}")
    print(f"ARCHIVE_DIR        : {ARCHIVE_DIR}")
    print(f"DRY_RUN            : {DRY_RUN}")
    print(f"ARCHIVE_OVERWRITE  : {ARCHIVE_OVERWRITE}")
    print(f"DEL_PREV_REPORT        : {DEL_PREV_REPORT}")
    print("------------------------------------------------------------")

    previous_shapefiles = find_shapefiles(PREVIOUS_WORKSPACE)
    current_shapefiles = find_shapefiles(RAW_WORKSPACE)

    previous_index = build_shapefile_index(previous_shapefiles)

    print(f"Previous shapefiles found: {len(previous_shapefiles):,}")
    print(f"Current shapefiles found : {len(current_shapefiles):,}")
    print("------------------------------------------------------------")

    report_rows: list[dict[str, Any]] = []

    for idx, current_shp in enumerate(current_shapefiles, start=1):
        previous_matches = previous_index.get(current_shp.stem, [])

        print(f"[{idx:,}/{len(current_shapefiles):,}] {current_shp}")
        print(f"  Previous match count: {len(previous_matches)}")

        result = process_current_shapefile(
            current_shp=current_shp,
            previous_shp_paths=previous_matches,
        )

        print(
            f"  {result.get('status', '')} | "
            f"Delete={result.get('Count of Delete', 0)} | "
            f"Notice={result.get('Count of Notice', 0)} | "
            f"{result.get('message', '')}"
        )

        report_rows.append(result)

    print("------------------------------------------------------------")
    print("Checking empty shapefiles after delete process")

    # Rescan after deleting rows. Some shapefiles may now have 0 polygons.
    remaining_current_shapefiles = find_shapefiles(RAW_WORKSPACE)

    empty_rows: list[dict[str, Any]] = []
    delete_rows: list[dict[str, Any]] = []
    archive_rows: list[dict[str, Any]] = []
    copied_stems: set[str] = set()

    for idx, shp_path in enumerate(remaining_current_shapefiles, start=1):
        empty_result = check_empty_shapefile(shp_path)
        empty_rows.append(empty_result)

        print(
            f"  EMPTY CHECK [{idx:,}/{len(remaining_current_shapefiles):,}] "
            f"{shp_path.stem} | {empty_result.get('status', '')} | "
            f"count={empty_result.get('feature_count', '')}"
        )

        if empty_result.get("is_empty") == "YES":
            # No polygons left at all -> nothing worth keeping, delete it outright.
            delete_result = delete_empty_shapefile_group(shp_path)
            delete_rows.append(delete_result)

            print(
                f"    DELETE EMPTY | {delete_result.get('status', '')} | "
                f"files={delete_result.get('deleted file count', 0)} | "
                f"{delete_result.get('message', '')}"
            )
        else:
            # Still has polygons -> this is the cleaned, final data for this round,
            # so it gets archived. RAW_WORKSPACE is left untouched (copy, not move).
            archive_result = archive_surviving_shapefile_group(
                shp_path=shp_path,
                copied_stems=copied_stems,
            )
            archive_rows.append(archive_result)

            print(
                f"    ARCHIVE | {archive_result.get('status', '')} | "
                f"files={archive_result.get('copied file count', 0)} | "
                f"{archive_result.get('message', '')}"
            )

    print("------------------------------------------------------------")

    total_deleted_polygons = sum(int(row.get("Count of Delete") or 0) for row in report_rows)
    empty_count = sum(1 for row in empty_rows if row.get("is_empty") == "YES")
    survivor_count = len(empty_rows) - empty_count
    archived_count = sum(1 for row in archive_rows if row.get("status") in ("COPIED", "DRY_RUN_COPY"))

    print(f"Total duplicate polygons deleted this run : {total_deleted_polygons:,}")
    print(f"Empty shapefiles deleted from RAW_WORKSPACE: {empty_count:,}")
    print(f"Surviving shapefiles copied to ARCHIVE_DIR  : {archived_count:,} (of {survivor_count:,} eligible)")

    if archived_count < survivor_count:
        print(
            "NOTE: Some surviving shapefiles were not copied - check the 'archive_survivors' "
            "sheet 'status'/'message' columns (common cause: ARCHIVE_OVERWRITE = False and a "
            "matching shapefile group already exists in ARCHIVE_DIR)."
        )

    write_report_excel(
        report_rows=report_rows,
        empty_rows=empty_rows,
        delete_rows=delete_rows,
        archive_rows=archive_rows,
    )

    print("============================================================")
    print(f"Report written: {DEL_PREV_REPORT}")

    if DRY_RUN:
        print("DRY_RUN = True, so no polygons were deleted, NOTICE was not written, empty shapefiles were not deleted, and surviving shapefiles were not copied.")
        print("After checking the Excel report, set DRY_RUN = False and run again.")
    else:
        print("Processing completed. Exact duplicates were deleted, NOTICE was written, empty shapefiles were deleted, and surviving shapefiles were copied to archive.")


if __name__ == "__main__":
    main()


Delete previous duplicated polygons / write NOTICE / delete empty shapefiles / archive survivors
Previous_STAMP     : 2026-01
ROUNDSTAMP         : 2026-06
PREVIOUS_WORKSPACE : D:\A02-Projects\Shapefile_Cleaning\02_archive\2026-01
RAW_WORKSPACE      : D:\A02-Projects\Shapefile_Cleaning\00_raw\2026-06
ARCHIVE_DIR        : D:\A02-Projects\Shapefile_Cleaning\02_archive\2026-06
DRY_RUN            : False
ARCHIVE_OVERWRITE  : False
DEL_PREV_REPORT        : D:\A02-Projects\Shapefile_Cleaning\05_reports\2026-06\04_2026-06-Del_Previous.xlsx
------------------------------------------------------------
Previous shapefiles found: 480
Current shapefiles found : 12
------------------------------------------------------------
[1/12] D:\A02-Projects\Shapefile_Cleaning\00_raw\2026-06\36000000\36000000_47.shp
  Previous match count: 1
  SUCCESS | Delete=20 | Notice=0 | Deleted exact duplicates and wrote NOTICE. NOTICE field status=ADDED. Skipped null geometry=0. Previous keys loaded
[2/12] D:\A02-Projec